# 1 - FWI gradient by automatic differentiation

Forward-model a 2D acoustic plate, then get the misfit gradient `dJ/d(alpha2)` for free with `loss.backward()`, and run an autodiff-driven inversion. Ported from PhD MATLAB NDT code.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seidlr/acoustic-fwi-ndt-pytorch/blob/main/notebooks/01_autodiff_fwi.ipynb)

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Replace OWNER with the GitHub account hosting this repo. This clones the repo
    # (which includes the domain data files) and installs the package editable.
    OWNER = "seidlr"
    !git clone -q https://github.com/{OWNER}/acoustic-fwi-ndt-pytorch.git
    %cd acoustic-fwi-ndt-pytorch
    !pip install -q -e .

import torch
from IPython.display import Image, display
from fwi.config import resolve_device, resolve_dtype

device = resolve_device()
dtype = resolve_dtype(device)
print("device:", device, "| dtype:", dtype)
print("Note: clean float64 needs CPU or CUDA; Apple MPS runs float32 (see caveat below).")

## Forward modeling
Build the small problem (homogeneous start model, 1-defect true model) and look at a wavefield snapshot. Autodiff defaults to the small grid (a full 104x304x800 autograd graph is memory-heavy).

In [ ]:
from fwi.problems import build_problem
from fwi.forward import forward
from fwi import plotting

prob = build_problem('small', device=device, dtype=dtype)
res = forward(prob.true_alpha2, prob.src_sig, prob.src_i, prob.src_j,
              prob.rec_i, prob.rec_j, prob.cfg, capture_wavefield=True)
p = plotting.save_field(res.wavefield[prob.cfg.nt//3], 'outputs/nb1_wavefield.png',
                        title='wavefield snapshot')
display(Image(str(p)))

## Autodiff gradient
`alpha2.requires_grad_(True)` -> forward -> misfit -> `backward()`.

In [ ]:
from fwi.misfit import autodiff_gradient

grad, J = autodiff_gradient(prob.start_alpha2, prob.observed, prob.src_sig,
    prob.src_i, prob.src_j, prob.rec_i, prob.rec_j, prob.cfg,
    active_mask=prob.active_mask)
print(f'misfit J = {J:.3e}')
display(Image(str(plotting.save_field(grad, 'outputs/nb1_kernel.png',
    title='autodiff dJ/d(alpha2)'))))

## Autodiff-driven inversion
Reconstruct the defect from the observed traces.

In [ ]:
from fwi.inversion import invert

alpha2_hat, history = invert(prob.start_alpha2, prob.observed, prob.src_sig,
    prob.src_i, prob.src_j, prob.rec_i, prob.rec_j, prob.cfg,
    active_mask=prob.active_mask, grad_fn='autodiff', n_iter=60)
print(f'misfit {history[0]:.3e} -> {history[-1]:.3e} ({history[0]/history[-1]:.1f}x)')
display(Image(str(plotting.save_convergence(history, 'outputs/nb1_conv.png'))))
display(Image(str(plotting.save_field(alpha2_hat - prob.start_alpha2,
    'outputs/nb1_update.png', title='recovered update'))))